In [0]:
from functools import reduce
from pyspark.sql.functions import col, lit, regexp_replace, trim, when

carpeta_base = "/Volumes/workspace/investigacion/datos_csv"

# columnas con valores numéricos (los archivos con separador ';' usan coma decimal)
columnas_numericas = ["CANTI", "PBK", "PNK", "FOBDOL", "FOBPES", "AGRENA", "FLETES", "SEGURO", "OTROSG"]

for anio in range(2015, 2026):

    # listar solo los .csv (ignora archivos como desktop.ini)
    archivos = [f.path for f in dbutils.fs.ls(f"{carpeta_base}/{anio}")
                if f.path.lower().endswith(".csv")]

    # los archivos del DANE mezclan separadores: ',' en unos y ';' en otros
    por_separador = {",": [], ";": []}
    for archivo in archivos:
        primera_linea = dbutils.fs.head(archivo, 500).splitlines()[0]
        sep = ";" if primera_linea.count(";") >= primera_linea.count(",") else ","
        por_separador[sep].append(archivo)

    dfs = []
    for sep, rutas in por_separador.items():
        if not rutas:
            continue
        df_sep = (spark.read
                  .option("header", True)
                  .option("sep", sep)
                  .csv(rutas))
        # limpiar nombres de columna (BOM, espacios, retornos de carro) y unificar tipos:
        # se lee todo como texto y solo las numéricas se convierten a double
        df_sep = df_sep.toDF(*[c.strip().lstrip("\ufeff").strip() for c in df_sep.columns])
        for c in columnas_numericas:
            if c in df_sep.columns:
                if sep == ";":
                    # formato con ';' como separador: puntos de miles y coma decimal
                    # (ej: ' 5.091.734,64 ' -> 5091734.64)
                    limpio = regexp_replace(
                        regexp_replace(
                            regexp_replace(col(c), "\\s", ""),
                            "\\.", ""),
                        ",", ".")
                else:
                    # formato con ',' como separador: punto decimal directo
                    limpio = trim(col(c))
                # '-' y cadenas vacías son marcadores de dato faltante en los CSV del DANE
                limpio = when(trim(limpio).isin("", "-"), lit(None)).otherwise(limpio)
                df_sep = df_sep.withColumn(c, limpio.cast("double"))
        dfs.append(df_sep)

    # desde 2022 el DANE ya no publica NIT ni RAZ_SIAL: se alinean las columnas
    # faltantes con null antes de unir (esta versión de PySpark no admite
    # allowMissMatchingColumns en unionByName)
    todas_columnas = []
    for d in dfs:
        for c in d.columns:
            if c not in todas_columnas:
                todas_columnas.append(c)
    dfs = [d.select(*[col(c) if c in d.columns else lit(None).cast("string").alias(c)
                      for c in todas_columnas])
           for d in dfs]
    df = reduce(lambda a, b: a.unionByName(b), dfs)

    (df.write
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"workspace.investigacion.datos_{anio}"))

    print(f"Datos del año {anio} cargados correctamente")

In [0]:
ruta = "/Volumes/workspace/investigacion/datos_csv/2015/*.csv"

df_2015 = (spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(ruta))

display(df_2015.limit(10))

In [0]:
df_2015.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.investigacion.datos_2015")

In [0]:
from functools import reduce
from pyspark.sql.functions import col, lit

for anio in range(2016, 2026):

    carpeta = f"/Volumes/workspace/investigacion/datos_csv/{anio}"

    # los archivos del DANE mezclan separadores: ',' en unos y ';' en otros,
    # así que se detecta el de cada archivo leyendo su encabezado
    por_separador = {",": [], ";": []}
    for f in dbutils.fs.ls(carpeta):
        if not f.path.lower().endswith(".csv"):
            continue
        primera_linea = dbutils.fs.head(f.path, 500).splitlines()[0]
        sep = ";" if primera_linea.count(";") >= primera_linea.count(",") else ","
        por_separador[sep].append(f.path)

    dfs = []
    for sep, rutas in por_separador.items():
        if not rutas:
            continue
        df_sep = (spark.read
            .option("header", True)
            .option("sep", sep)
            .csv(rutas))
        # limpiar los nombres de columna (comillas, BOM, espacios)
        df_sep = df_sep.toDF(*[c.strip().lstrip("\ufeff").strip() for c in df_sep.columns])
        dfs.append(df_sep)

    # desde 2022 el DANE ya no publica NIT ni RAZ_SIAL: se alinean
    # las columnas faltantes con null antes de unir
    todas_columnas = []
    for d in dfs:
        for c in d.columns:
            if c not in todas_columnas:
                todas_columnas.append(c)
    dfs = [d.select(*[col(c) if c in d.columns else lit(None).cast("string").alias(c)
                      for c in todas_columnas])
           for d in dfs]
    df = reduce(lambda a, b: a.unionByName(b), dfs)

    (df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"workspace.investigacion.datos_{anio}"))

    print(f"Tabla datos_{anio} creada correctamente")

In [0]:
for anio in range(2015, 2026):

    tabla = f"workspace.investigacion.datos_{anio}"

    df = spark.table(tabla)

    print(f"AÑO: {anio}")
    print(f"Registros: {df.count()}")
    print(f"Columnas: {len(df.columns)}")
    print("-" * 40)

In [0]:
for anio in [2015, 2022, 2023, 2025]:

    df = spark.table(
        f"workspace.investigacion.datos_{anio}"
    )

    print(f"\n========== {anio} ==========")

    for columna in df.columns:
        print(columna)

In [0]:
df = spark.table("workspace.investigacion.datos_2025")

display(df.limit(10))

In [0]:
df = spark.table("workspace.investigacion.datos_2025")

print("TIPOS DE DATOS:")
df.printSchema()

print("EJEMPLOS DE FECHAS Y VALORES:")
display(
    df.select("FECH", "FOBDOL", "FOBPES", "PAIS", "POSAR")
      .limit(15)
)

In [0]:
# Tendencia actual
from pyspark.sql.functions import col, lit, sum as sum, trim, when, regexp_replace
from functools import reduce
import matplotlib.pyplot as plt

# Unir los datos de los 11 años
tablas = []

for anio in range(2015, 2026):
    df = spark.table(f"workspace.investigacion.datos_{anio}")

    # Convertir FOBDOL a número
    valor = trim(col("FOBDOL").cast("string"))
    # si tiene coma es el formato DANE con ';' (puntos de miles + coma
    # decimal): se quitan los puntos de miles y la coma pasa a punto decimal
    valor = when(
        valor.contains(","),
        regexp_replace(regexp_replace(valor, "\\.", ""), ",", ".")
    ).otherwise(valor)
    # '-' y vacío son marcadores de dato faltante en los CSV del DANE
    valor = when(valor.isin("", "-"), lit(None)).otherwise(valor)

    df = df.withColumn("FOB_NUM", valor.cast("double"))
    df = df.withColumn("ANIO", lit(anio))

    tablas.append(df.select("ANIO", "FOB_NUM"))

datos = reduce(lambda a, b: a.unionByName(b), tablas)

# Sumar exportaciones por año
anual = (datos
    .groupBy("ANIO")
    .agg(sum("FOB_NUM").alias("EXPORTACIONES_USD"))
    .orderBy("ANIO")
)

# Pasar el resultado a pandas para graficar
resultado_anual = anual.toPandas()

plt.figure(figsize=(12, 5))
plt.plot(
    resultado_anual["ANIO"],
    resultado_anual["EXPORTACIONES_USD"] / 1_000_000,
    marker="o"
)
plt.title("Evolución anual de las exportaciones de Colombia (2015–2025)")
plt.xlabel("Año")
plt.ylabel("Valor FOB (millones de dólares)")
plt.xticks(resultado_anual["ANIO"])
plt.grid(True)
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import col, lit, sum as sum, trim, when, regexp_replace
from functools import reduce
import matplotlib.pyplot as plt

# Exportaciones originadas en Risaralda (código DANE 66 en DPTO1)
tablas_risaralda = []

for anio in range(2015, 2026):
    df = spark.table(f"workspace.investigacion.datos_{anio}")

    # Convertir FOBDOL a número (mismo formato mixto del DANE)
    valor = trim(col("FOBDOL").cast("string"))
    valor = when(
        valor.contains(","),
        regexp_replace(regexp_replace(valor, "\\.", ""), ",", ".")
    ).otherwise(valor)
    valor = when(valor.isin("", "-"), lit(None)).otherwise(valor)

    df = df.withColumn("FOB_NUM", valor.cast("double"))
    df = df.withColumn("ANIO", lit(anio))

    tablas_risaralda.append(
        df.filter(col("DPTO1") == "66").select("ANIO", "FOB_NUM")
    )

datos_risaralda = reduce(lambda a, b: a.unionByName(b), tablas_risaralda)

# Sumar exportaciones de Risaralda por año
anual_risaralda = (datos_risaralda
    .groupBy("ANIO")
    .agg(sum("FOB_NUM").alias("EXPORTACIONES_USD"))
    .orderBy("ANIO")
)

resultado_risaralda = anual_risaralda.toPandas()

plt.figure(figsize=(12, 5))
plt.plot(
    resultado_risaralda["ANIO"],
    resultado_risaralda["EXPORTACIONES_USD"] / 1_000_000,
    marker="o"
)
plt.title("Evolución anual de las exportaciones de Risaralda (2015–2025)")
plt.xlabel("Año")
plt.ylabel("Valor FOB (millones de dólares)")
plt.xticks(resultado_risaralda["ANIO"])
plt.grid(True)
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import to_date, date_format, concat, lit, when

tablas_mensuales = []

for anio in range(2015, 2026):
    df = spark.table(f"workspace.investigacion.datos_{anio}")

    # Convertir FOBDOL a número
    valor = trim(col("FOBDOL").cast("string"))
    # si tiene coma es el formato DANE con ';' (puntos de miles + coma
    # decimal): se quitan los puntos de miles y la coma pasa a punto decimal
    valor = when(
        valor.contains(","),
        regexp_replace(regexp_replace(valor, "\\.", ""), ",", ".")
    ).otherwise(valor)
    # '-' y vacío son marcadores de dato faltante en los CSV del DANE
    valor = when(valor.isin("", "-"), lit(None)).otherwise(valor)

    df = df.withColumn("FOB_NUM", valor.cast("double"))

    # FECH es un código YYMM (ej: 1504 = abril de 2015), no una fecha
    # completa: se convierte en fecha y las filas con FECH inválido quedan
    # en null (después el filtro isNotNull las excluye)
    fecha = trim(col("FECH").cast("string"))
    df = df.withColumn(
        "FECHA",
        when(fecha.rlike("^\\d{4}$"),
             to_date(concat(lit("20"), fecha, lit("01")), "yyyyMMdd"))
    )

    tablas_mensuales.append(
        df.select("FECHA", "FOB_NUM")
    )

datos_mensuales = reduce(
    lambda a, b: a.unionByName(b),
    tablas_mensuales
)

# Agrupar las exportaciones por año y mes
mensual = (datos_mensuales
    .filter(col("FECHA").isNotNull())
    .groupBy(date_format("FECHA", "yyyy-MM").alias("PERIODO"))
    .agg(sum("FOB_NUM").alias("EXPORTACIONES_USD"))
    .orderBy("PERIODO")
)

resultado_mensual = mensual.toPandas()

In [0]:
from pyspark.sql.functions import col, lit, sum as sum, trim, when, regexp_replace, to_date, date_format, concat
from functools import reduce
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# Exportaciones mensuales originadas en Risaralda (código DANE 66 en DPTO1).
# Nota: estos archivos del DANE no traen municipio, solo departamento:
# Risaralda es la aproximación disponible para Pereira.
tablas_mensuales_pereira = []

for anio in range(2015, 2026):
    df = spark.table(f"workspace.investigacion.datos_{anio}")

    # Convertir FOBDOL a número (formato mixto del DANE)
    valor = trim(col("FOBDOL").cast("string"))
    valor = when(
        valor.contains(","),
        regexp_replace(regexp_replace(valor, "\\.", ""), ",", ".")
    ).otherwise(valor)
    valor = when(valor.isin("", "-"), lit(None)).otherwise(valor)

    df = df.withColumn("FOB_NUM", valor.cast("double"))

    # FECH es un código YYMM (ej: 1504 = abril de 2015)
    fecha = trim(col("FECH").cast("string"))
    df = df.withColumn(
        "FECHA",
        when(fecha.rlike("^\\d{4}$"),
             to_date(concat(lit("20"), fecha, lit("01")), "yyyyMMdd"))
    )

    tablas_mensuales_pereira.append(
        df.filter(col("DPTO1") == "66").select("FECHA", "FOB_NUM")
    )

datos_mensuales_pereira = reduce(
    lambda a, b: a.unionByName(b),
    tablas_mensuales_pereira
)

mensual_pereira = (datos_mensuales_pereira
    .filter(col("FECHA").isNotNull())
    .groupBy(date_format("FECHA", "yyyy-MM").alias("PERIODO"))
    .agg(sum("FOB_NUM").alias("EXPORTACIONES_USD"))
    .orderBy("PERIODO")
)

resultado_mensual_pereira = mensual_pereira.toPandas()
resultado_mensual_pereira["PERIODO"] = pd.to_datetime(resultado_mensual_pereira["PERIODO"])

plt.figure(figsize=(15, 6))
plt.plot(
    resultado_mensual_pereira["PERIODO"],
    resultado_mensual_pereira["EXPORTACIONES_USD"] / 1_000_000
)
plt.title("Evolución mensual de las exportaciones de Pereira/Risaralda (2015–2025)")
plt.xlabel("Periodo")
plt.ylabel("Valor FOB (millones de dólares)")
plt.grid(True)
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import col, lit, lpad, sum as spark_sum, trim, when, regexp_replace
from functools import reduce
import matplotlib.pyplot as plt

# Top de productos exportados originados en Risaralda (código DANE 66 en DPTO1)
tablas_productos_risaralda = []

for anio in range(2015, 2026):
    df = spark.table(f"workspace.investigacion.datos_{anio}")

    valor = trim(col("FOBDOL").cast("string"))
    # si tiene coma es el formato DANE con ';' (puntos de miles + coma
    # decimal): se quitan los puntos de miles y la coma pasa a punto decimal
    valor = when(
        valor.contains(","),
        regexp_replace(regexp_replace(valor, "\\.", ""), ",", ".")
    ).otherwise(valor)
    # '-' y vacío son marcadores de dato faltante en los CSV del DANE
    valor = when(valor.isin("", "-"), lit(None)).otherwise(valor)

    df = df.withColumn("FOB_NUM", valor.cast("double"))

    # lpad rellena con ceros a 10 dígitos: los archivos nuevos del DANE
    # pierden el cero inicial del código (ej: 901119000 = 0901119000)
    tablas_productos_risaralda.append(
        df.filter(col("DPTO1") == "66").select(
            lpad(col("POSAR"), 10, "0").alias("POSAR"), "FOB_NUM")
    )

datos_productos_risaralda = reduce(
    lambda a, b: a.unionByName(b),
    tablas_productos_risaralda
)

productos_risaralda = (
    datos_productos_risaralda
    .filter(col("POSAR").isNotNull() & col("FOB_NUM").isNotNull())
    .groupBy("POSAR")
    .agg(spark_sum("FOB_NUM").alias("TOTAL_USD"))
    .orderBy(col("TOTAL_USD").desc())
)

top_productos_risaralda = productos_risaralda.limit(10).toPandas()
display(productos_risaralda.limit(10))

plt.figure(figsize=(12, 6))
plt.barh(
    top_productos_risaralda["POSAR"].astype(str),
    top_productos_risaralda["TOTAL_USD"] / 1_000_000
)

plt.title("Top 10 de productos exportados por Risaralda (2015–2025)")
plt.xlabel("Valor exportado (millones de dólares)")
plt.ylabel("Posición arancelaria")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import col, lit, sum as spark_sum, trim, when, regexp_replace
from functools import reduce
import matplotlib.pyplot as plt

# Top de destinos de las exportaciones originadas en Risaralda (DPTO1 = 66).
# Se agrupa por COD_PAI4 (siglas de país) porque el código numérico de PAIS
# cambió con los años (EE.UU. pasó de 249 a 840 desde 2025) y quedaría duplicado.
tablas_destinos_risaralda = []

for anio in range(2015, 2026):
    df = spark.table(f"workspace.investigacion.datos_{anio}")

    valor = trim(col("FOBDOL").cast("string"))
    # si tiene coma es el formato DANE con ';' (puntos de miles + coma
    # decimal): se quitan los puntos de miles y la coma pasa a punto decimal
    valor = when(
        valor.contains(","),
        regexp_replace(regexp_replace(valor, "\\.", ""), ",", ".")
    ).otherwise(valor)
    # '-' y vacío son marcadores de dato faltante en los CSV del DANE
    valor = when(valor.isin("", "-"), lit(None)).otherwise(valor)

    df = df.withColumn("FOB_NUM", valor.cast("double"))

    tablas_destinos_risaralda.append(
        df.filter(col("DPTO1") == "66").select("COD_PAI4", "FOB_NUM")
    )

datos_destinos_risaralda = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    tablas_destinos_risaralda
)

destinos_risaralda = (
    datos_destinos_risaralda
    .filter(col("COD_PAI4").isNotNull() & col("FOB_NUM").isNotNull())
    .groupBy("COD_PAI4")
    .agg(spark_sum("FOB_NUM").alias("TOTAL_USD"))
    .orderBy(col("TOTAL_USD").desc())
)

top_destinos_risaralda = destinos_risaralda.limit(10).toPandas()
display(destinos_risaralda.limit(10))

plt.figure(figsize=(12, 6))
plt.barh(
    top_destinos_risaralda["COD_PAI4"].astype(str),
    top_destinos_risaralda["TOTAL_USD"] / 1_000_000
)

plt.title("Top 10 de destinos comerciales de Risaralda (2015–2025)")
plt.xlabel("Valor exportado (millones de dólares)")
plt.ylabel("País de destino")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

resultado_mensual["PERIODO"] = pd.to_datetime(
    resultado_mensual["PERIODO"]
)

plt.figure(figsize=(15, 6))
plt.plot(
    resultado_mensual["PERIODO"],
    resultado_mensual["EXPORTACIONES_USD"] / 1_000_000
)

plt.title("Evolución mensual de las exportaciones de Colombia (2015–2025)")
plt.xlabel("Periodo")
plt.ylabel("Valor FOB (millones de dólares)")
plt.grid(True)

plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import col, lit, sum as spark_sum, trim, when, regexp_replace
from functools import reduce

tablas_productos = []

for anio in range(2015, 2026):
    df = spark.table(f"workspace.investigacion.datos_{anio}")

    valor = trim(col("FOBDOL").cast("string"))
    # si tiene coma es el formato DANE con ';' (puntos de miles + coma
    # decimal): se quitan los puntos de miles y la coma pasa a punto decimal
    valor = when(
        valor.contains(","),
        regexp_replace(regexp_replace(valor, "\\.", ""), ",", ".")
    ).otherwise(valor)
    # '-' y vacío son marcadores de dato faltante en los CSV del DANE
    valor = when(valor.isin("", "-"), lit(None)).otherwise(valor)

    df = df.withColumn("FOB_NUM", valor.cast("double"))

    tablas_productos.append(
        df.select("POSAR", "FOB_NUM")
    )

datos_productos = reduce(
    lambda a, b: a.unionByName(b),
    tablas_productos
)

productos = (
    datos_productos
    .filter(col("POSAR").isNotNull() & col("FOB_NUM").isNotNull())
    .groupBy("POSAR")
    .agg(spark_sum("FOB_NUM").alias("TOTAL_USD"))
    .orderBy(col("TOTAL_USD").desc())
)

top_productos = productos.limit(10).toPandas()
display(productos.limit(10))

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.barh(
    top_productos["POSAR"].astype(str),
    top_productos["TOTAL_USD"] / 1_000_000
)

plt.title("Top 10 de productos exportados por Colombia (2015–2025)")
plt.xlabel("Valor exportado (millones de dólares)")
plt.ylabel("Posición arancelaria")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import col, lit, sum as spark_sum, trim, when, regexp_replace
from functools import reduce

tablas_destinos = []

for anio in range(2015, 2026):
    df = spark.table(f"workspace.investigacion.datos_{anio}")

    valor = trim(col("FOBDOL").cast("string"))
    # si tiene coma es el formato DANE con ';' (puntos de miles + coma
    # decimal): se quitan los puntos de miles y la coma pasa a punto decimal
    valor = when(
        valor.contains(","),
        regexp_replace(regexp_replace(valor, "\\.", ""), ",", ".")
    ).otherwise(valor)
    # '-' y vacío son marcadores de dato faltante en los CSV del DANE
    valor = when(valor.isin("", "-"), lit(None)).otherwise(valor)

    df = df.withColumn("FOB_NUM", valor.cast("double"))

    tablas_destinos.append(
        df.select("PAIS", "FOB_NUM")
    )

datos_destinos = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    tablas_destinos
)

destinos = (
    datos_destinos
    .filter(col("PAIS").isNotNull() & col("FOB_NUM").isNotNull())
    .groupBy("PAIS")
    .agg(spark_sum("FOB_NUM").alias("TOTAL_USD"))
    .orderBy(col("TOTAL_USD").desc())
)

top_destinos = destinos.limit(10).toPandas()
display(destinos.limit(10))

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

plt.barh(
    top_destinos["PAIS"].astype(str),
    top_destinos["TOTAL_USD"] / 1_000_000
)

plt.title("Top 10 de destinos comerciales de Colombia (2015–2025)")
plt.xlabel("Valor exportado (millones de dólares)")
plt.ylabel("País de destino")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()


In [0]:
from pyspark.sql.functions import col, lit, sum as sum, trim, when, regexp_replace
from functools import reduce
import matplotlib.pyplot as plt

# Exportaciones originadas en Risaralda (código DANE 66 en DPTO1)
tablas_risaralda = []

for anio in range(2015, 2026):
    df = spark.table(f"workspace.investigacion.datos_{anio}")

    # Convertir FOBDOL a número (mismo formato mixto del DANE)
    valor = trim(col("FOBDOL").cast("string"))
    valor = when(
        valor.contains(","),
        regexp_replace(regexp_replace(valor, "\\.", ""), ",", ".")
    ).otherwise(valor)
    valor = when(valor.isin("", "-"), lit(None)).otherwise(valor)

    df = df.withColumn("FOB_NUM", valor.cast("double"))
    df = df.withColumn("ANIO", lit(anio))

    tablas_risaralda.append(
        df.filter(col("DPTO1") == "66").select("ANIO", "FOB_NUM")
    )

datos_risaralda = reduce(lambda a, b: a.unionByName(b), tablas_risaralda)

# Sumar exportaciones de Risaralda por año
anual_risaralda = (datos_risaralda
    .groupBy("ANIO")
    .agg(sum("FOB_NUM").alias("EXPORTACIONES_USD"))
    .orderBy("ANIO")
)

resultado_risaralda = anual_risaralda.toPandas()

plt.figure(figsize=(12, 5))
plt.plot(
    resultado_risaralda["ANIO"],
    resultado_risaralda["EXPORTACIONES_USD"] / 1_000_000,
    marker="o"
)
plt.title("Evolución anual de las exportaciones de Risaralda (2015–2025)")
plt.xlabel("Año")
plt.ylabel("Valor FOB (millones de dólares)")
plt.xticks(resultado_risaralda["ANIO"])
plt.grid(True)
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import col, lit, sum as sum, trim, when, regexp_replace, to_date, date_format, concat
from functools import reduce
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd

# Exportaciones mensuales originadas en Risaralda (código DANE 66 en DPTO1).
# Nota: estos archivos del DANE no traen municipio, solo departamento:
# Risaralda es la aproximación disponible para Pereira.
tablas_mensuales_pereira = []

for anio in range(2015, 2026):
    df = spark.table(f"workspace.investigacion.datos_{anio}")

    # Convertir FOBDOL a número (formato mixto del DANE)
    valor = trim(col("FOBDOL").cast("string"))
    valor = when(
        valor.contains(","),
        regexp_replace(regexp_replace(valor, "\\.", ""), ",", ".")
    ).otherwise(valor)
    valor = when(valor.isin("", "-"), lit(None)).otherwise(valor)

    df = df.withColumn("FOB_NUM", valor.cast("double"))

    # FECH es un código YYMM (ej: 1504 = abril de 2015)
    fecha = trim(col("FECH").cast("string"))
    df = df.withColumn(
        "FECHA",
        when(fecha.rlike("^\\d{4}$"),
             to_date(concat(lit("20"), fecha, lit("01")), "yyyyMMdd"))
    )

    tablas_mensuales_pereira.append(
        df.filter(col("DPTO1") == "66").select("FECHA", "FOB_NUM")
    )

datos_mensuales_pereira = reduce(
    lambda a, b: a.unionByName(b),
    tablas_mensuales_pereira
)

mensual_pereira = (datos_mensuales_pereira
    .filter(col("FECHA").isNotNull())
    .groupBy(date_format("FECHA", "yyyy-MM").alias("PERIODO"))
    .agg(sum("FOB_NUM").alias("EXPORTACIONES_USD"))
    .orderBy("PERIODO")
)

resultado_mensual_pereira = mensual_pereira.toPandas()
resultado_mensual_pereira["PERIODO"] = pd.to_datetime(resultado_mensual_pereira["PERIODO"])

plt.figure(figsize=(15, 6))
plt.plot(
    resultado_mensual_pereira["PERIODO"],
    resultado_mensual_pereira["EXPORTACIONES_USD"] / 1_000_000
)
plt.title("Evolución mensual de las exportaciones de Pereira/Risaralda (2015–2025)")
plt.xlabel("Periodo")
plt.ylabel("Valor FOB (millones de dólares)")
plt.grid(True)
plt.gca().xaxis.set_major_locator(mdates.YearLocator())
plt.gca().xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import col, lit, lpad, sum as spark_sum, trim, when, regexp_replace
from functools import reduce
import matplotlib.pyplot as plt

# Top de productos exportados originados en Risaralda (código DANE 66 en DPTO1)
tablas_productos_risaralda = []

for anio in range(2015, 2026):
    df = spark.table(f"workspace.investigacion.datos_{anio}")

    valor = trim(col("FOBDOL").cast("string"))
    # si tiene coma es el formato DANE con ';' (puntos de miles + coma
    # decimal): se quitan los puntos de miles y la coma pasa a punto decimal
    valor = when(
        valor.contains(","),
        regexp_replace(regexp_replace(valor, "\\.", ""), ",", ".")
    ).otherwise(valor)
    # '-' y vacío son marcadores de dato faltante en los CSV del DANE
    valor = when(valor.isin("", "-"), lit(None)).otherwise(valor)

    df = df.withColumn("FOB_NUM", valor.cast("double"))

    # lpad rellena con ceros a 10 dígitos: los archivos nuevos del DANE
    # pierden el cero inicial del código (ej: 901119000 = 0901119000)
    tablas_productos_risaralda.append(
        df.filter(col("DPTO1") == "66").select(
            lpad(col("POSAR"), 10, "0").alias("POSAR"), "FOB_NUM")
    )

datos_productos_risaralda = reduce(
    lambda a, b: a.unionByName(b),
    tablas_productos_risaralda
)

productos_risaralda = (
    datos_productos_risaralda
    .filter(col("POSAR").isNotNull() & col("FOB_NUM").isNotNull())
    .groupBy("POSAR")
    .agg(spark_sum("FOB_NUM").alias("TOTAL_USD"))
    .orderBy(col("TOTAL_USD").desc())
)

top_productos_risaralda = productos_risaralda.limit(10).toPandas()
display(productos_risaralda.limit(10))

plt.figure(figsize=(12, 6))
plt.barh(
    top_productos_risaralda["POSAR"].astype(str),
    top_productos_risaralda["TOTAL_USD"] / 1_000_000
)

plt.title("Top 10 de productos exportados por Risaralda (2015–2025)")
plt.xlabel("Valor exportado (millones de dólares)")
plt.ylabel("Posición arancelaria")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

In [0]:
from pyspark.sql.functions import col, lit, sum as spark_sum, trim, when, regexp_replace
from functools import reduce
import matplotlib.pyplot as plt

# Top de destinos de las exportaciones originadas en Risaralda (DPTO1 = 66).
# Se agrupa por COD_PAI4 (siglas de país) porque el código numérico de PAIS
# cambió con los años (EE.UU. pasó de 249 a 840 desde 2025) y quedaría duplicado.
tablas_destinos_risaralda = []

for anio in range(2015, 2026):
    df = spark.table(f"workspace.investigacion.datos_{anio}")

    valor = trim(col("FOBDOL").cast("string"))
    # si tiene coma es el formato DANE con ';' (puntos de miles + coma
    # decimal): se quitan los puntos de miles y la coma pasa a punto decimal
    valor = when(
        valor.contains(","),
        regexp_replace(regexp_replace(valor, "\\.", ""), ",", ".")
    ).otherwise(valor)
    # '-' y vacío son marcadores de dato faltante en los CSV del DANE
    valor = when(valor.isin("", "-"), lit(None)).otherwise(valor)

    df = df.withColumn("FOB_NUM", valor.cast("double"))

    tablas_destinos_risaralda.append(
        df.filter(col("DPTO1") == "66").select("COD_PAI4", "FOB_NUM")
    )

datos_destinos_risaralda = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    tablas_destinos_risaralda
)

destinos_risaralda = (
    datos_destinos_risaralda
    .filter(col("COD_PAI4").isNotNull() & col("FOB_NUM").isNotNull())
    .groupBy("COD_PAI4")
    .agg(spark_sum("FOB_NUM").alias("TOTAL_USD"))
    .orderBy(col("TOTAL_USD").desc())
)

top_destinos_risaralda = destinos_risaralda.limit(10).toPandas()
display(destinos_risaralda.limit(10))

plt.figure(figsize=(12, 6))
plt.barh(
    top_destinos_risaralda["COD_PAI4"].astype(str),
    top_destinos_risaralda["TOTAL_USD"] / 1_000_000
)

plt.title("Top 10 de destinos comerciales de Risaralda (2015–2025)")
plt.xlabel("Valor exportado (millones de dólares)")
plt.ylabel("País de destino")
plt.gca().invert_yaxis()
plt.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()